# Week 13: Multimodal AI
**Applied Generative AI**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain vision–language architectures** — How models like CLIP, BLIP/BLIP-2, and LLaVA connect images and text (contrastive learning, captioning, instruction tuning).
2. **Implement image understanding** — Zero-shot classification with CLIP, captioning and visual question answering (VQA) with BLIP from Hugging Face.
3. **Build audio→language pipelines** — Transcribe speech with Whisper and chain the transcript to text analysis or summarization.
4. **Use multimodal APIs responsibly** — Call OpenAI vision-capable models with `getpass` for secrets and compare API outputs to open-source baselines.
5. **Evaluate multimodal outputs** — Reason about caption quality metrics (e.g., BLEU, CIDEr), human evaluation, and common failure modes (hallucinations, bias, accessibility).
6. **Identify risks** — Connect multimodal deployment (health, education, surveillance) to bias, representation, and governance questions.

---
## Setup — environment (Colab-friendly)

This notebook uses **open-source models on Hugging Face** (CLIP, BLIP, Whisper) so most sections need **no API key**. The OpenAI section uses `getpass` so your key is not stored in the notebook file.

**Recommended:** Runtime → **GPU** (T4 or better) in Colab for faster inference. CPU works but is slower for BLIP/Whisper.

In [ ]:
%pip install -q transformers torch torchvision Pillow requests openai librosa soundfile accelerate datasets

In [ ]:
import io
import json
import base64
import warnings
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
from PIL import Image

warnings.filterwarnings("ignore", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def load_image_from_url(url: str, timeout: int = 30) -> Image.Image:
    """Download an image from a URL into a PIL RGB image."""
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return Image.open(io.BytesIO(r.content)).convert("RGB")

---
## 1. The multimodal AI landscape

### Vision + language
- **GPT-4V / GPT-4o** — General-purpose multimodal chat: image + text in, text (and sometimes audio) out; strong on open-ended description and reasoning.
- **LLaVA** — Open, instruction-tuned vision–language assistant built on a frozen vision encoder + LLM; popular for research and local deployment.
- **CLIP** — Contrastive image–text encoder; excels at **zero-shot** alignment (image↔text similarity) but does not generate captions by itself.
- **BLIP / BLIP-2** — Captioning, VQA, and multimodal understanding; BLIP-2 uses a Q-Former to connect frozen encoders to an LLM.

### Audio + language
- **Whisper** — Robust speech-to-text; often chained with an LLM for summarization, extraction, or translation.

### Video
- **Emerging models** — Video understanding often samples frames + audio and fuses with transformers; the field is moving quickly (e.g., video LLMs, multimodal diffusion).

### Architecture patterns
| Pattern | Idea | Examples |
|--------|------|----------|
| **Early fusion** | Combine raw or shallow features from modalities up front | Some classic multimodal nets |
| **Late fusion** | Encode each modality separately, merge scores or embeddings at the end | CLIP-style dual encoders + classifier head |
| **Cross-attention** | One modality attends over another’s sequence of tokens | BLIP-2 Q-Former, many VL transformers |

---
## 2. Image understanding with CLIP (open source)

**CLIP** learns a shared embedding space for images and text. We use **zero-shot classification** by scoring each label string against the image and taking the softmax over logits.

Below we load `openai/clip-vit-base-patch32` from Hugging Face (no API key).

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_id).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_id)
clip_model.eval()
print("CLIP loaded:", clip_id)

In [ ]:
@torch.inference_mode()
def clip_zero_shot_classify(image: Image.Image, candidate_labels: List[str]) -> dict:
    """Return label probabilities from CLIP image-text matching."""
    inputs = clip_processor(
        text=candidate_labels,
        images=image,
        return_tensors="pt",
        padding=True,
    ).to(device)
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1).squeeze(0).tolist()
    ranked = sorted(zip(candidate_labels, probs), key=lambda x: x[1], reverse=True)
    return {"ranked": ranked, "top": ranked[0]}


# Public-domain style photo (Unsplash); swap URL if offline
img_url = "https://images.unsplash.com/photo-1543466835-00a7907e9de1?w=640"
zebra_img = load_image_from_url(img_url)

labels = [
    "a photo of a zebra",
    "a photo of a dog",
    "a photo of a car",
    "a photo of food on a plate",
    "a photo of a city skyline",
]
result = clip_zero_shot_classify(zebra_img, labels)
print("Top prediction:", result["top"])
for lab, p in result["ranked"]:
    print(f"  {p:.3f}  {lab}")

plt.figure(figsize=(6, 4))
plt.imshow(zebra_img)
plt.axis("off")
plt.title("Sample image (URL)")
plt.show()

In [ ]:
# Simple synthetic image: red square on white background (matplotlib)
fig, ax = plt.subplots(figsize=(3, 3))
ax.set_facecolor("white")
rect = plt.Rectangle((0.25, 0.25), 0.5, 0.5, facecolor="red", edgecolor="black")
ax.add_patch(rect)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
plt.tight_layout()
buf = io.BytesIO()
plt.savefig(buf, format="png", bbox_inches="tight", pad_inches=0)
plt.close(fig)
buf.seek(0)
synthetic = Image.open(buf).convert("RGB")

syn_labels = [
    "a red square on a white background",
    "a blue circle",
    "a photograph of a cat",
    "a geometric abstract composition",
]
syn_result = clip_zero_shot_classify(synthetic, syn_labels)
print("Synthetic image — top:", syn_result["top"])
plt.imshow(synthetic)
plt.axis("off")
plt.show()

In [ ]:
@torch.inference_mode()
def clip_image_text_similarity(image: Image.Image, texts: List[str]) -> List[Tuple[str, float]]:
    """Cosine-like CLIP scores (not guaranteed exactly cosine; use for ranking)."""
    inputs = clip_processor(
        text=texts,
        images=image,
        return_tensors="pt",
        padding=True,
    ).to(device)
    out = clip_model(**inputs)
    # logits_per_image: (1, num_texts)
    scores = out.logits_per_image.squeeze(0).tolist()
    return list(zip(texts, scores))


captions = [
    "A zebra standing in tall grass.",
    "A dog catching a frisbee at the beach.",
    "An empty parking lot at night.",
]
pairs = clip_image_text_similarity(zebra_img, captions)
pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)
print("Caption ↔ image CLIP scores (higher = better match under CLIP):")
for c, s in pairs_sorted:
    print(f"  {s:8.3f}  {c}")

---
## 3. Image captioning & visual QA with BLIP (open source)

We use **BLIP** (`Salesforce/blip-image-captioning-base`) for unconditional and **conditional** captioning, and **BLIP-VQA** (`Salesforce/blip-vqa-base`) for question answering about pixels.

BLIP-2 is stronger but heavier; on CPU-only Colab, BLIP is a more reliable default.

In [ ]:
from transformers import BlipForConditionalGeneration, BlipProcessor

blip_cap_id = "Salesforce/blip-image-captioning-base"
blip_processor = BlipProcessor.from_pretrained(blip_cap_id)
blip_caption_model = BlipForConditionalGeneration.from_pretrained(blip_cap_id).to(device)
blip_caption_model.eval()


@torch.inference_mode()
def blip_caption(image: Image.Image, prompt: Optional[str] = None, max_new_tokens: int = 50) -> str:
    if prompt:
        inputs = blip_processor(image, prompt, return_tensors="pt").to(device)
    else:
        inputs = blip_processor(image, return_tensors="pt").to(device)
    out = blip_caption_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return blip_processor.decode(out[0], skip_special_tokens=True)


cap_default = blip_caption(zebra_img)
print("Unconditional caption:", cap_default)

cap_photo = blip_caption(zebra_img, prompt="a photography of")
cap_art = blip_caption(zebra_img, prompt="a watercolor painting of")
print('Conditional — "a photography of":', cap_photo)
print('Conditional — "a watercolor painting of":', cap_art)

In [ ]:
# Compare captions on a second diverse image (objects / indoor)
img2_url = "https://images.unsplash.com/photo-1511707171634-5f897ff02aa9?w=640"
phone_img = load_image_from_url(img2_url)

for prompt in [None, "this is an advertisement showing", "a technical diagram of"]:
    ptxt = "(unconditional)" if prompt is None else repr(prompt)
    print(ptxt, "->", blip_caption(phone_img, prompt=prompt))

plt.imshow(phone_img)
plt.axis("off")
plt.title("Second sample image")
plt.show()

In [ ]:
from transformers import BlipForQuestionAnswering

blip_vqa_id = "Salesforce/blip-vqa-base"
vqa_processor = BlipProcessor.from_pretrained(blip_vqa_id)
vqa_model = BlipForQuestionAnswering.from_pretrained(blip_vqa_id).to(device)
vqa_model.eval()


@torch.inference_mode()
def visual_qa(image: Image.Image, question: str) -> str:
    inputs = vqa_processor(image, question, return_tensors="pt").to(device)
    out = vqa_model.generate(**inputs)
    return vqa_processor.decode(out[0], skip_special_tokens=True)


questions = [
    "What animal is in the picture?",
    "Is the background mostly grass?",
    "What is the main color of the animal?",
]
for q in questions:
    print("Q:", q)
    print("A:", visual_qa(zebra_img, q))
    print("---")

---
## 4. Audio–language pipeline: Whisper → text → analysis

We load **Whisper** from Hugging Face for **ASR** (automatic speech recognition), then pass the transcript into a small **Hugging Face summarization** model so this section stays **API-free**.

In production you might swap the second stage for GPT-4o or another LLM.

In [ ]:
import librosa
from datasets import load_dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor

whisper_id = "openai/whisper-base"
wprocessor = WhisperProcessor.from_pretrained(whisper_id)
whisper_model = WhisperForConditionalGeneration.from_pretrained(whisper_id).to(device)
whisper_model.eval()

# Short speech clip via Hugging Face `datasets` (reliable in Colab; no manual URL)
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
ex = ds[0]
waveform = np.asarray(ex["audio"]["array"], dtype=np.float32)
sr_native = int(ex["audio"]["sampling_rate"])
if sr_native != 16000:
    waveform = librosa.resample(waveform, orig_sr=sr_native, target_sr=16000)
sr = 16000
print("Sample rate:", sr, "| duration (s):", float(waveform.shape[0]) / sr)

# Optional: listen in Colab
try:
    from IPython.display import Audio

    display(Audio(waveform, rate=sr))
except Exception as e:
    print("Audio display skipped:", e)

In [ ]:
@torch.inference_mode()
def whisper_transcribe(waveform_16k: np.ndarray) -> str:
    inputs = wprocessor(
        waveform_16k,
        sampling_rate=16000,
        return_tensors="pt",
    ).input_features.to(device)
    # Whisper expects forced decoder ids for English by default in many tutorials;
    # here we use model.generate with default language behavior.
    ids = whisper_model.generate(inputs)
    text = wprocessor.batch_decode(ids, skip_special_tokens=True)[0]
    return text.strip()


transcript = whisper_transcribe(waveform)
print("Transcript:\n", transcript)

In [ ]:
from transformers import pipeline

# Lightweight summarization on CPU/GPU (no OpenAI key)
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0 if torch.cuda.is_available() else -1,
)

def summarize_transcript(text: str, max_words: int = 60) -> str:
    text = text.strip()
    if len(text.split()) < 12:
        return "Transcript too short for abstractive summarization — returning raw text.\n" + text
    # BART CNN works best on multi-sentence news-ish text; clip very long inputs
    chunk = text[:2500]
    out = summarizer(chunk, max_length=min(130, max_words + 20), min_length=12, do_sample=False)
    return out[0]["summary_text"]


summary = summarize_transcript(transcript)
print("Summary:\n", summary)

In [ ]:
def audio_to_text_analysis(waveform_16k: np.ndarray) -> dict:
    """End-to-end demo: ASR + extractive-style signals + summarization."""
    t = whisper_transcribe(waveform_16k)
    words = t.split()
    analysis = {
        "transcript": t,
        "char_count": len(t),
        "word_count": len(words),
        "summary": summarize_transcript(t),
    }
    return analysis


report = audio_to_text_analysis(waveform)
print(json.dumps({k: v for k, v in report.items() if k != "transcript"}, indent=2))
print("\nTranscript (first 400 chars):\n", report["transcript"][:400])

---
## 5. Multimodal understanding with the OpenAI API

Vision-capable chat models (e.g. **GPT-4o**, **GPT-4o-mini**) accept an **image URL** or a **base64 data URL** plus a text prompt. Below we use `getpass` so the key is not pasted into the notebook source.

**Prerequisites:** `pip install openai` (included in setup) and a valid API key with model access.

**Note:** Run the **CLIP** section above first so `zebra_img` is defined, or set `zebra_img` to any `PIL.Image.Image` before calling the API cell.

In [ ]:
import os
from getpass import getpass

from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key (input hidden): ")

oai_client = OpenAI()
print("OpenAI client ready.")

In [ ]:
def image_to_data_url(image: Image.Image, fmt: str = "JPEG") -> str:
    buf = io.BytesIO()
    image.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    mime = "image/jpeg" if fmt.upper() == "JPEG" else "image/png"
    return f"data:{mime};base64,{b64}"


vision_model = "gpt-4o-mini"  # vision-capable; change if your account uses different defaults

data_url = image_to_data_url(zebra_img.resize((512, 512)))

user_prompt = (
    "Return a JSON object with keys: "
    "subject (string), setting (string), colors (list of up to 5 strings), "
    "confidence (string: low/medium/high), caveats (string: what you cannot verify). "
    "No markdown fences — JSON only."
)

resp = oai_client.chat.completions.create(
    model=vision_model,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }
    ],
    max_tokens=400,
    temperature=0.2,
)

api_text = resp.choices[0].message.content
print("--- API structured response (raw) ---")
print(api_text)

try:
    parsed = json.loads(api_text)
    print("\n--- Parsed JSON ---")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print("\n(Model did not return strict JSON; inspect raw text above.)")

### Comparing API vs open-source outputs

| Axis | CLIP / BLIP (local HF) | GPT-4o / mini (API) |
|------|-------------------------|---------------------|
| **Strengths** | Free to run, reproducible, good for similarity & caption baselines | Flexible instructions, strong world knowledge, structured outputs |
| **Weaknesses** | Fixed model behavior; BLIP can be generic | Cost, latency, policy filters, key management |
| **Trust & eval** | You control weights & prompts | Must log prompts/responses for audit; version drift when OpenAI updates models |

**Try this:** Run BLIP’s caption on `zebra_img` and compare to the API JSON’s `subject` / `setting` — note omissions, hedging, and any attributes neither model can verify (e.g., exact species sub-type).

---
## 6. Evaluating multimodal outputs

### Reference-based caption metrics (concepts)
- **BLEU** — N-gram overlap between candidate and one or more reference captions; fast but brittle (synonyms penalized).
- **CIDEr** — TF-IDF weighted n-gram consensus with human references; common on COCO captioning benchmarks.
- **CLIPScore** — Use a frozen CLIP to score image–caption pairs; reference-free and correlates reasonably with human judgments for *grounding*, not fluency.

In assignments you often report **multiple metrics** plus **qualitative examples**.

### Human evaluation (framework)
1. **Grounding** — Does every object/relationship in the caption appear in the image?
2. **Coverage** — Are salient regions/attributes mentioned?
3. **Harm** — Stereotypes, sensitive attributes inferred without evidence?
4. **Usefulness** — Fit for the downstream task (accessibility alt text vs. marketing copy).

### Common failure modes
- **Object hallucination** — Mentioning things not present (often from language priors).
- **Missed fine detail** — Small text, distant objects, rare categories.
- **Cultural bias** — Default assumptions about dress, roles, or context.
- **Accessibility gaps** — Poor performance on low light, blur, or assistive use cases unless explicitly engineered.

In [ ]:
@torch.inference_mode()
def clipscore_style(image: Image.Image, caption: str) -> float:
    """Single scalar CLIP similarity (image vs one caption). Higher ≈ more aligned under CLIP."""
    inputs = clip_processor(
        text=[caption],
        images=image,
        return_tensors="pt",
        padding=True,
    ).to(device)
    out = clip_model(**inputs)
    # scale logit as a crude score (not official CLIPScore scaling)
    return float(out.logits_per_image.squeeze().item())


candidate_captions = [
    blip_caption(zebra_img),
    "A zebra grazing in green grass with trees in the background.",
    "A red sports car parked in front of a café.",
]

print("CLIP-style alignment scores (ranking aid, not a full metric suite):")
for c in candidate_captions:
    print(f"{clipscore_style(zebra_img, c):8.3f}  {c}")

print("\nManual checklist for the BLIP caption:")
print("- Grounding: tick each noun phrase against pixels.")
print("- Coverage: list 3 salient traits BLIP skipped.")
print("- Failure scan: any unwarranted gender/age/culture guesses?")

In [ ]:
# Optional: install sacrebleu in Colab if you want reference-based BLEU on your own references
# %pip install -q sacrebleu

def human_eval_rubric() -> dict:
    """Return a printable rubric for annotators (1–5 Likert per axis)."""
    return {
        "grounding": "1 = many hallucinations … 5 = fully supported by image",
        "coverage": "1 = misses main content … 5 = main salient content captured",
        "fluency": "1 = ungrammatical … 5 = natural language",
        "bias_safety": "1 = problematic assumptions … 5 = neutral / evidence-based",
    }


print(json.dumps(human_eval_rubric(), indent=2))

---
## 7. Exercises (submit according to course instructions)

**Exercise 1 — CLIP zero-shot classifier**  
Pick **8–12 custom categories** (e.g., your own photo collection themes). Use CLIP to classify **at least 5 images** (URLs or local files). Report the top-1 and top-3 accuracy *against your own labels* and discuss one misclassification.

**Exercise 2 — Caption + manual evaluation**  
Generate captions for **five diverse images** (scene, object, people, text-in-image, low light). For each, write **3–5 bullet points** using the grounding/coverage/bias checklist above.

**Exercise 3 — Audio analysis pipeline**  
Record or download a **30–60s** speech clip (appropriate license). Run Whisper + your choice of second stage (HF summarization or OpenAI). Submit the transcript, the analysis/summary, and one limitation you observed (noise, accent, domain vocabulary, etc.).

---
## 8. Responsible AI reflection

Multimodal AI systems can “see” and “hear,” but their perception is shaped by training data that reflects specific cultural contexts, demographics, and perspectives. Vision models perform worse on underrepresented groups; captioning models default to stereotypical descriptions. As multimodal AI moves into healthcare (medical imaging), education (accessibility tools), and surveillance, **what safeguards are needed?** **Who should decide** what these systems are allowed to perceive and describe?

Use your discussion post or report to connect this prompt to **concrete policies** (consent, opt-out, human review, disclosure, red-teaming for bias) rather than only abstract ethics.

---
## 9. Summary & next steps

- **CLIP** gives strong **zero-shot** image–text scoring; **BLIP** adds **generation** and **VQA**.
- **Whisper** is the standard open **ASR** backbone; pair it with summarization or LLM reasoning for full **audio→language** workflows.
- **API vision models** are flexible for structured outputs; **open weights** help with reproducibility and offline use — choose based on cost, privacy, and governance.
- **Evaluation** should combine **automated signals** (CLIPScore, BLEU/CIDEr where references exist) with **human rubrics** and **failure-mode checklists**.

**Next week preview:** connect multimodal perception to **agents** and **tool use** (when to invoke vision/ASR tools, how to verify tool outputs, and how to document multimodal decisions for responsible deployment).